In [ ]:
import logging
import os
import sys
import traceback
from tqdm import tqdm 

from hydra import initialize, compose
import numpy as np
import pandas as pd
from omegaconf import DictConfig
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import torch

import sys 
sys.path.insert(0, "../../../../../shared_utils")
from experiment_utils import get_forward_model

from torch.utils.data import TensorDataset, DataLoader

In [6]:
with initialize(config_path="../../../../../inverse/loss_guidance/config", version_base=None):
    config = compose(config_name="run_inverse",
                    overrides=[
                        "paths=bloodplus"
                    ])

In [7]:
# config["paths"]["ct_classifier_path"] = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/checkpoints/cell_type_classifier/2026-04-08_00-00-38/easy-monkey-341_TargetPredictionModel.pkl" 

In [8]:
logger = logging.getLogger(__name__)

forward_model, (
    flow_matching,
    target_prediction_model
) = get_forward_model(config, logger=logger)

In [9]:
# Classify both training and validation anndata 

In [10]:
train_data_phi = flow_matching.train_data.state_data
valid_data_phi = flow_matching.validation_data['random_state=42-split_to_retrieve=0-test_size=0.3-K=3'].state_data

In [11]:
annot_train = np.array(flow_matching.train_data.adata.obs.index)
annot_valid = np.array(flow_matching.validation_data['random_state=42-split_to_retrieve=0-test_size=0.3-K=3'].adata.obs.index)

In [12]:
X_concat = torch.from_numpy(np.concatenate([train_data_phi, valid_data_phi], axis=0))
annot_concat = np.concatenate([annot_train, annot_valid])

## Tensor dataset for data 

In [13]:
target_prediction = forward_model.target_prediction_model
target_prediction.eval()

RescaledTargetPredictionModel(
  (resc_model): ModuleDict(
    (inv_params): IZNorm(
      (params): ModuleDict(
        (mean): ModelParam()
        (covariance): ModelParam()
      )
    )
    (fwd_params): ZNorm(
      (params): ModuleDict(
        (mean): ModelParam()
        (covariance): ModelParam()
      )
    )
    (model): PerturbationApproximatePosterior(
      (pert_approximate_posterior): ModuleDict(
        (encoder): MLPBlock(
          (net): Sequential(
            (0): Sequential(
              (0): Linear(in_features=26, out_features=128, bias=True)
              (1): ReLU()
              (2): Dropout(p=0.0, inplace=False)
            )
            (1): Sequential(
              (0): Linear(in_features=128, out_features=128, bias=True)
              (1): ReLU()
              (2): Dropout(p=0.0, inplace=False)
            )
            (2): Sequential(
              (0): Linear(in_features=128, out_features=64, bias=True)
              (1): Identity()
            )
  

In [14]:
dataloader = DataLoader(
    torch.utils.data.TensorDataset(X_concat), 
    batch_size=4026,
    shuffle=False
)

In [15]:
ct_predictions = []

with torch.no_grad():
    for batch in tqdm(dataloader):
        X_batch = batch[0].to(target_prediction_model.device)
        y_pred = target_prediction(X_batch)["cell_type"].argmax(1)
        ct_predictions.append(y_pred.cpu().detach().numpy())

100%|██████████| 12938/12938 [04:08<00:00, 52.10it/s]


In [16]:
ct_predictions = np.concatenate(ct_predictions)

In [17]:
from sklearn.preprocessing import LabelEncoder

# Prepare label encoder
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = np.array(ct_le.classes_.tolist())

In [18]:
ct_predictions_str = classes[ct_predictions]

In [19]:
pd.DataFrame(ct_predictions_str).value_counts()

0          
MPP            8101034
GMP-Neutro     7960072
MgKEryPro1     7085116
LMPP-CLP       6533733
Early_GMP      4329486
GMP_cycling    4052824
EosBasMast2    3291886
EosBasMast1    2174121
MPP_MgkEry     2106143
MgKEryPro2     1770105
EryPro         1484568
CyclingPro1     930617
late_MgkPro     654521
pre_cDC2        348140
earlyMPP        228550
preProB         198050
MLP             179255
CD16Mono        142507
CD14Mono        129886
HSCs            127829
EosBasMast3      93095
CyclingPro2      80250
pre_cDC1         68762
ProB             15215
Name: count, dtype: int64

In [20]:
ct_predictions_str.shape

(52085765,)

In [21]:
annot_concat.shape

(52085765,)

In [22]:
X_concat.shape

torch.Size([52085765, 26])

Read and annotate real data 

In [23]:
adata_real = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/new_dataset/BloodPlus_Logicle_pm_.h5ad")

In [24]:
adata_real.obs.loc[annot_concat, "cell_type"] = ct_predictions_str

In [25]:
# adata_real_subset = sc.pp.subsample(adata_real, n_obs=100_000,  random_state=42, copy=True)

In [26]:
# sc.tl.pca(adata_real_subset)
# sc.pp.neighbors(adata_real_subset)
# sc.tl.umap(adata_real_subset)

In [27]:
# sc.pl.umap(adata_real_subset, color="cell_type", legend_loc="on data", s=20)

In [28]:
# adata_real_subset = adata_real_subset[:, :20]

In [29]:
# sc.tl.pca(adata_real_subset)
# sc.pp.neighbors(adata_real_subset)
# sc.tl.umap(adata_real_subset)

In [30]:
# sc.pl.umap(adata_real_subset, color="cell_type", legend_loc="on data", s=20)

# Perform protocol analysis

In [31]:
# Cell types to keep 
to_keep = ["CD14Mono",
            "CD16Mono",
            "CyclingPro1",
            "CyclingPro2",
            "EosBasMast1",
            "EosBasMast2",
            "EosBasMast3",
            "EryPro",
            "HSCs",
            "ProB",
            "late_MgkPro",
            'GMP-Neutro',
            'preProB',
            'pre_cDC1']

In [32]:
protocol_columns = ["740-yp_[µm]", 
                   "mtg_[µm]", 
                   "rhflt3l_[ng_ml]", 
                   "gm-csf_[ng_ml]", 
                   "tpo_[ng_ml]",
                   "sr1_[nm]", 
                   "um171_[nm]", 
                   "um729_[µm]", 
                   "scf_[ng_ml]",
                   "butyzamide_[nm]",
                   "retinoic_acid_[µm]",
                   "ldl_[ng_ml]", 
                   "il3_[ng_ml]", 
                   "o2_[%]", 
                   "days_of_culture"]

We will create an AnnData with unique protocols as `.obs` dimensions

In [ ]:
# Unique protocols 
obs_protocols = adata_real.obs.loc[:, protocol_columns].values.astype(float)
X_protocols = np.unique(obs_protocols, axis=0)

In [44]:
X_protocols

array([[  5., 100.,  10., ...,   0.,  20.,  13.],
       [  5., 100.,  10., ...,   0.,  20.,  15.],
       [  5., 100.,  10., ...,   0.,  20.,  21.],
       ...,
       [  5., 100.,  10., ...,   0.,  20.,  21.],
       [  5., 100.,  10., ...,   0.,  20.,  14.],
       [  5., 100.,  10., ...,   0.,  20.,  21.]], shape=(103, 15))

In [45]:
obs_protocols

array([[  5., 100.,  10., ...,   0.,  20.,  13.],
       [  5., 100.,  10., ...,   0.,  20.,  13.],
       [  5., 100.,  10., ...,   0.,  20.,  13.],
       ...,
       [  5., 100.,  10., ...,   0.,  20.,  14.],
       [  5., 100.,  10., ...,   0.,  20.,  14.],
       [  5., 100.,  10., ...,   0.,  20.,  14.]], shape=(52085765, 15))

Create hashes 

In [ ]:
protocol_hashes = {}
for prot_number, protocol in enumerate(X_protocols):
    # Derive string from protocol 
    protocol_str = "_".join(list(protocol.astype(str)))
    # Hash it using the corresponding integer
    protocol_hashes[prot_number] = protocol_str

iprotocol_hases = {val:key for key,val in protocol_hashes.items()}

In [40]:
real_data_prot_id = []

for i in tqdm(range(obs_protocols.shape[0])):
    protocol_i_str = "_".join(obs_protocols[i].astype("str"))
    protocol_i_hash = iprotocol_hases[protocol_i_str]
    real_data_prot_id.append(protocol_i_hash)

100%|██████████| 52085765/52085765 [10:44<00:00, 80766.23it/s]


In [41]:
real_data_prot_id = np.array(real_data_prot_id)

In [42]:
adata_real.obs["protocol_hash"] = real_data_prot_id

In [43]:
# Obs 
cell_type_proportions = {cell_type: [] for cell_type in adata_real.obs.cell_type.unique()}
cell_type_counts = []

for i, unique_protocol in tqdm(enumerate(X_protocols)):
    # Hash index of the protocol 
    idx = adata_real.obs.protocol_hash == i
    # Data with that protocol 
    adata_protocol = adata_real[idx]
    proportions = adata_protocol.obs.cell_type.value_counts() / adata_protocol.shape[0]
    
    for cell_type in cell_type_proportions: 
        if cell_type in proportions.index.values:
            cell_type_proportions[cell_type].append(proportions[cell_type])
        else:
            cell_type_proportions[cell_type].append(0.0)

    # List representing the protocol 
    unique_protocol_list = list(unique_protocol)
    unique_protocol_list += [adata_protocol.shape[0]]
    cell_type_counts.append(unique_protocol_list)

103it [01:24,  1.22it/s]


In [50]:
np.array(cell_type_counts)

array([[5.00000e+00, 1.00000e+02, 1.00000e+01, ..., 2.00000e+01,
        1.30000e+01, 9.22790e+04],
       [5.00000e+00, 1.00000e+02, 1.00000e+01, ..., 2.00000e+01,
        1.50000e+01, 1.53850e+04],
       [5.00000e+00, 1.00000e+02, 1.00000e+01, ..., 2.00000e+01,
        2.10000e+01, 1.63930e+04],
       ...,
       [5.00000e+00, 1.00000e+02, 1.00000e+01, ..., 2.00000e+01,
        2.10000e+01, 5.80711e+05],
       [5.00000e+00, 1.00000e+02, 1.00000e+01, ..., 2.00000e+01,
        1.40000e+01, 1.63979e+05],
       [5.00000e+00, 1.00000e+02, 1.00000e+01, ..., 2.00000e+01,
        2.10000e+01, 1.91691e+05]], shape=(103, 16))

In [53]:
colnames = protocol_columns + ["cell_counts"]

In [55]:
cell_count_protocol_df = pd.DataFrame(cell_type_counts)
cell_count_protocol_df.columns = colnames

In [58]:
cell_count_protocol_df["cell_counts"] = np.log(cell_count_protocol_df["cell_counts"])

In [60]:
cell_count_protocol_df.to_csv("cell_count_protocol_df.csv")

## Check results

In [ ]:
proportion_df = pd.DataFrame(cell_type_proportions)

protocol_adata = sc.AnnData(X=np.log1p(X_protocols), obs=proportion_df)
protocol_adata.var.index = protocol_columns

In [ ]:
argmax_protocols = np.argmax(protocol_adata.obs, axis=0)

In [ ]:
adata_argmax = protocol_adata[argmax_protocols]
adata_argmax.obs["cell_type"] = np.array(adata_argmax.obs.columns)
adata_argmax.obs.drop(["cell_type"], axis=1).max(0)

In [ ]:
adata_argmax.obs.drop(["cell_type"], axis=1).max(0).loc[to_keep]

Max enriched proportions for the 500k dataset 

CD14Mono       0.007501 \
CD16Mono       0.043888 \
CyclingPro1    0.704545 \
CyclingPro2    0.032233 \
EosBasMast1    0.334009 \
EosBasMast2    0.408742 \
EosBasMast3    0.037866 \
EryPro         0.316938 \
HSCs           0.068761 \
ProB           0.002957 \
late_MgkPro    0.715066 \
GMP-Neutro     0.665340 \
preProB        0.010091 \
pre_cDC2       0.064441 

# Check distribution of proportions per cell type

In [ ]:
df = protocol_adata.obs.melt()

plt.figure(figsize=(5, 5))

sns.stripplot(
    data=df,
    x="variable",
    y="value",
    # showfliers=False  # cleaner look (optional)
)

plt.xticks(rotation=90)  # rotate labels
plt.xlabel("")
plt.ylabel("Proportions per protocol")
# plt.title("Proportions per protocols (103 unique protocols)", fontsize=14)

plt.tight_layout()
plt.show()

## Save adata 

In [ ]:
protocol_adata.obs

In [ ]:
protocol_adata.obs.max(0)

In [ ]:
protocol_adata.write_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/miscellaneous/unique_concentrations_adata_bloodplus.h5ad")